# 01 — Data Collection & Preprocessing

**Proyecto:** Global Life Expectancy Prediction with Machine Learning  
**Autor:** Marcelo Corro  
**Fuentes:** WHO GHO API · World Bank API  

Este notebook documenta el pipeline de obtención y preprocesamiento de datos:

1. Descarga de indicadores desde WHO GHO API y World Bank API
2. Exploración de los datos crudos
3. Merge, limpieza e imputación
4. Validación del dataset final

> Los scripts `src/collect_data.py` y `src/preprocess.py` se invocan automáticamente si los archivos de datos no existen.

In [1]:
import os
import subprocess
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Asegurar que trabajamos desde la raiz del proyecto
if Path.cwd().name == 'notebooks':
    os.chdir('..')

ROOT       = Path.cwd()
RAW_DIR    = ROOT / 'data' / 'raw'
PROC_DIR   = ROOT / 'data' / 'processed'

print(f'Directorio raiz: {ROOT}')
print(f'Python version: {pd.__version__} (pandas)')

Directorio raiz: C:\ChlenixProjects\life-expectancy-ml
Python version: 3.0.3 (pandas)


---
## 1. Obtención de Datos

### Fuentes

| Fuente | Indicadores | Acceso |
|--------|------------|--------|
| **WHO GHO** | Esperanza de vida, mortalidad, vacunacion, medicos | REST API |
| **World Bank** | PIB, salud, educacion, agua, electricidad, internet | `wbgapi.fetch()` |

**Periodo:** 2000–2022 · **Cobertura:** ~190 paises

In [2]:
who_raw  = RAW_DIR / 'who_indicators.csv'
wb_raw   = RAW_DIR / 'wb_indicators.csv'
final    = PROC_DIR / 'dataset_final.csv'

if not who_raw.exists() or not wb_raw.exists():
    print('Descargando datos crudos...')
    subprocess.run(['python', 'src/collect_data.py'], check=True)
else:
    print('Datos crudos ya disponibles.')

if not final.exists():
    print('Ejecutando preprocesamiento...')
    subprocess.run(['python', 'src/preprocess.py'], check=True)
else:
    print('Dataset final ya disponible.')

Datos crudos ya disponibles.
Dataset final ya disponible.


---
## 2. Datos Crudos — WHO GHO

In [3]:
who = pd.read_csv(who_raw)
print(f'Shape: {who.shape}')
print(f'Paises unicos: {who["iso3"].nunique()}')
print(f'Anios: {who["anio"].min()} - {who["anio"].max()}')
print(f'\nColumnas: {list(who.columns)}')
who.head()

Shape: (4622, 10)
Paises unicos: 201
Anios: 2000 - 2022

Columnas: ['iso3', 'anio', 'esperanza_vida', 'mortalidad_infantil', 'mortalidad_adulta', 'prevalencia_vih', 'vacunacion_dtp', 'vacunacion_hepatitis_b', 'medicos_por_10000', 'prevalencia_anemia_ninos']


,iso3,anio,esperanza_vida,mortalidad_infantil,mortalidad_adulta,prevalencia_vih,vacunacion_dtp,vacunacion_hepatitis_b,medicos_por_10000,prevalencia_anemia_ninos
0,AFG,2000,53.834299,131.632458,378.194110,1400.0,24.0,0.0,NaN,30.90
1,AFG,2001,53.921629,127.407600,380.338179,1600.0,33.0,0.0,2.02,29.88
2,AFG,2002,55.159352,123.112733,367.522126,1800.0,36.0,0.0,NaN,28.98
3,AFG,2003,56.092501,118.707077,353.380984,2000.0,41.0,0.0,NaN,28.26
4,AFG,2004,56.484653,114.206348,345.667035,2200.0,50.0,0.0,NaN,27.72


In [4]:
print('Valores faltantes por columna (WHO):')
nulos_who = who.isna().sum()
pct_who   = (nulos_who / len(who) * 100).round(1)
resumen_who = pd.DataFrame({'nulos': nulos_who, 'pct': pct_who})
resumen_who[resumen_who['nulos'] > 0].sort_values('pct', ascending=False)

Valores faltantes por columna (WHO):


,nulos,pct
medicos_por_10000,1773,38.4
prevalencia_vih,1221,26.4
prevalencia_anemia_ninos,722,15.6
esperanza_vida,552,11.9
mortalidad_adulta,552,11.9
vacunacion_hepatitis_b,179,3.9
vacunacion_dtp,179,3.9
mortalidad_infantil,22,0.5


---
## 3. Datos Crudos — World Bank

In [5]:
wb = pd.read_csv(wb_raw)
print(f'Shape: {wb.shape}')
print(f'Paises unicos: {wb["iso3"].nunique()}')
print(f'Anios: {wb["anio"].min()} - {wb["anio"].max()}')
print(f'\nColumnas: {list(wb.columns)}')
wb.head()

Shape: (6072, 11)
Paises unicos: 264
Anios: 2000 - 2022

Columnas: ['iso3', 'anio', 'pib_per_capita', 'gasto_salud_pct_pib', 'gasto_educacion_pct_pib', 'acceso_agua_potable', 'acceso_saneamiento', 'acceso_electricidad', 'tasa_desempleo', 'urbanizacion_pct', 'usuarios_internet_pct']


,iso3,anio,pib_per_capita,gasto_salud_pct_pib,gasto_educacion_pct_pib,acceso_agua_potable,acceso_saneamiento,acceso_electricidad,tasa_desempleo,urbanizacion_pct,usuarios_internet_pct
0,ABW,2000,20681.023027,NaN,4.71468,95.233536,97.973104,91.7,NaN,65.354550,15.442823
1,ABW,2001,20740.132583,NaN,4.79898,95.359056,98.012508,100.0,NaN,65.335114,17.100000
2,ABW,2002,21307.248251,NaN,4.87220,95.484576,98.051913,100.0,NaN,65.282069,18.800000
3,ABW,2003,21949.485996,NaN,NaN,95.610096,98.091317,100.0,NaN,65.198292,20.800000
4,ABW,2004,23700.631990,NaN,4.35699,95.735616,98.130722,100.0,NaN,65.088653,23.000000


In [6]:
print('Valores faltantes por columna (World Bank):')
nulos_wb = wb.isna().sum()
pct_wb   = (nulos_wb / len(wb) * 100).round(1)
resumen_wb = pd.DataFrame({'nulos': nulos_wb, 'pct': pct_wb})
resumen_wb[resumen_wb['nulos'] > 0].sort_values('pct', ascending=False)

Valores faltantes por columna (World Bank):


,nulos,pct
gasto_educacion_pct_pib,1937,31.9
usuarios_internet_pct,969,16.0
tasa_desempleo,691,11.4
gasto_salud_pct_pib,643,10.6
acceso_saneamiento,217,3.6
acceso_agua_potable,198,3.3
pib_per_capita,153,2.5
acceso_electricidad,71,1.2


---
## 4. Dataset Final

### Pipeline de preprocesamiento

1. Filtro ISO-3 — eliminar regiones/agregados que no son paises
2. Merge inner por `(iso3, anio)`
3. Eliminar filas sin target (`esperanza_vida`)
4. Imputacion por mediana de grupo `(region, anio)` y mediana global como fallback

In [7]:
df = pd.read_csv(final)
print(f'Shape: {df.shape}')
print(f'Paises unicos: {df["iso3"].nunique()}')
print(f'Anios: {df["anio"].min()} - {df["anio"].max()}')
print(f'Nulos totales: {df.isna().sum().sum()}')
df.head()

Shape: (4070, 21)
Paises unicos: 185
Anios: 2000 - 2021
Nulos totales: 0


,iso3,pais,region,anio,esperanza_vida,mortalidad_infantil,mortalidad_adulta,prevalencia_vih,vacunacion_dtp,vacunacion_hepatitis_b,...,prevalencia_anemia_ninos,pib_per_capita,gasto_salud_pct_pib,gasto_educacion_pct_pib,acceso_agua_potable,acceso_saneamiento,acceso_electricidad,tasa_desempleo,urbanizacion_pct,usuarios_internet_pct
0,AFG,Afghanistan,Asia,2000,53.834299,131.632458,378.194110,1400.0,24.0,0.0,...,30.90,174.930991,4.138679,3.460120,29.730966,22.298273,4.4,7.897,18.558200,1.183479
1,AFG,Afghanistan,Asia,2001,53.921629,127.407600,380.338179,1600.0,33.0,0.0,...,29.88,138.706822,4.290317,3.532130,29.762926,22.309525,9.3,7.973,18.741238,0.004723
2,AFG,Afghanistan,Asia,2002,55.159352,123.112733,367.522126,1800.0,36.0,0.0,...,28.98,178.954088,9.443391,3.777010,31.836077,23.734709,14.1,7.867,18.941248,0.004561
3,AFG,Afghanistan,Asia,2003,56.092501,118.707077,353.380984,2000.0,41.0,0.0,...,28.26,198.871116,8.941258,3.540025,33.908629,25.160117,19.0,7.844,19.160035,0.087891
4,AFG,Afghanistan,Asia,2004,56.484653,114.206348,345.667035,2200.0,50.0,0.0,...,27.72,221.763654,9.808474,3.455145,35.999355,26.592743,23.8,7.794,19.399359,0.105809


In [8]:
print('Tipos de datos:')
df.dtypes

Tipos de datos:


iso3                            str
pais                            str
region                          str
anio                          int64
esperanza_vida              float64
mortalidad_infantil         float64
mortalidad_adulta           float64
prevalencia_vih             float64
vacunacion_dtp              float64
vacunacion_hepatitis_b      float64
medicos_por_10000           float64
prevalencia_anemia_ninos    float64
pib_per_capita              float64
gasto_salud_pct_pib         float64
gasto_educacion_pct_pib     float64
acceso_agua_potable         float64
acceso_saneamiento          float64
acceso_electricidad         float64
tasa_desempleo              float64
urbanizacion_pct            float64
usuarios_internet_pct       float64
dtype: object

---
## 5. Distribución de la Variable Objetivo

In [9]:
print('Estadisticas de esperanza_vida:')
df['esperanza_vida'].describe().round(2)

Estadisticas de esperanza_vida:


count    4070.00
mean       70.17
std         8.62
min        40.16
25%        64.48
50%        72.31
75%        76.62
max        84.64
Name: esperanza_vida, dtype: float64

In [10]:
fig = px.histogram(
    df,
    x='esperanza_vida',
    nbins=40,
    color='region',
    title='Distribucion de Esperanza de Vida por Region (2000-2021)',
    labels={'esperanza_vida': 'Esperanza de Vida (anos)', 'count': 'Frecuencia'},
    template='plotly_white'
)
fig.show()

---
## 6. Cobertura por Región

In [11]:
cobertura = df.groupby('region').agg(
    paises=('iso3', 'nunique'),
    registros=('iso3', 'count'),
    ev_media=('esperanza_vida', 'mean'),
    ev_min=('esperanza_vida', 'min'),
    ev_max=('esperanza_vida', 'max')
).round(2).sort_values('paises', ascending=False)

print('Cobertura y estadisticas por region:')
cobertura

Cobertura y estadisticas por region:


,paises,registros,ev_media,ev_min,ev_max
region,,,,,
Africa,53,1166,60.60,41.80,77.53
Asia,47,1034,72.16,53.83,84.64
Europe,39,858,77.23,64.96,83.45
Americas,33,726,74.10,40.16,82.01
Oceania,10,220,69.37,60.13,83.37
Otro,3,66,76.00,72.10,80.69


---
## 7. Serie Temporal — Chile

In [12]:
chile = df[df['iso3'] == 'CHL'].sort_values('anio')

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=chile['anio'],
    y=chile['esperanza_vida'],
    mode='lines+markers',
    name='Chile',
    line=dict(color='#E63946', width=2.5)
))

latam_iso = ['ARG', 'BRA', 'COL', 'PER', 'MEX', 'BOL', 'ECU', 'URY', 'PRY', 'VEN']
latam_media = df[df['iso3'].isin(latam_iso)].groupby('anio')['esperanza_vida'].mean().reset_index()
fig.add_trace(go.Scatter(
    x=latam_media['anio'],
    y=latam_media['esperanza_vida'],
    mode='lines',
    name='Media LATAM',
    line=dict(color='#457B9D', width=1.5, dash='dash')
))

fig.update_layout(
    title='Esperanza de Vida: Chile vs Media LATAM (2000-2021)',
    xaxis_title='Ano',
    yaxis_title='Esperanza de Vida (anos)',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)
fig.show()

In [13]:
print('Chile — ultimos 5 registros:')
cols_muestra = ['anio', 'esperanza_vida', 'pib_per_capita', 'gasto_salud_pct_pib', 'acceso_agua_potable', 'medicos_por_10000']
chile[cols_muestra].tail(5).reset_index(drop=True)

Chile — ultimos 5 registros:


,anio,esperanza_vida,pib_per_capita,gasto_salud_pct_pib,acceso_agua_potable,medicos_por_10000
0,2017,80.636066,14879.908623,9.090518,98.348378,24.30
1,2018,80.939531,15659.480868,9.150404,98.456516,25.69
2,2019,81.010759,14496.934564,9.309685,98.564215,26.27
3,2020,79.907941,13117.993023,9.697346,98.671391,27.98
4,2021,79.045783,16216.184086,9.724502,98.778021,29.79


---
## 8. Resumen del Pipeline

| Paso | Resultado |
|------|-----------|
| Datos WHO crudos | 4.622 filas · 10 columnas |
| Datos World Bank crudos | 6.072 filas · 11 columnas |
| Merge inner + filtro ISO-3 | 4.507 filas |
| Eliminar sin target | 4.070 filas |
| Imputacion nulos | 0 nulos restantes |
| **Dataset final** | **4.070 filas · 185 paises · 22 anos · 16 features** |

**Siguiente paso:** `notebooks/02_eda.ipynb` — Analisis Exploratorio de Datos